# 共享 State：编译后的子图直接作节点

**场景：多智能体客服**。总控（父图）负责意图路由；"退款专员"是独立开发的子图，
内部有多步（生成回复 → 合规审查），但对父图来说它只是一个普通节点。
双方共享 `messages`——这正是官方文档给的标准场景：多智能体通过共享 `messages` 协作时，
把 Agent 图直接传给 `add_node`，同名通道自动双向读写，零包装代码。

| | 说明 |
|---|---|
| 前提 | 父/子图至少共享一个 state key |
| 写法 | `builder.add_node("节点名", 编译后的子图)` |
| 私有 key | 子图 schema 可以多出私有 key（如工单号），只在子图内部流转 |
| 版本注意 | langgraph 1.2.11 实测：**必须传编译后的图**，直接传未编译 builder 会 `TypeError` |

In [4]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Command
from rich import print
from typing import Annotated, Literal
from typing_extensions import TypedDict

load_dotenv(override=True)
model = init_chat_model("deepseek-flash")


class State(TypedDict):
    messages: Annotated[list, add_messages]


# ---- 子图：退款专员（内部两步，对父图只是一个节点）----
def draft_reply(state: State):
    user_msg = state["messages"][-1].content
    reply = model.invoke(
        f"你是退款专员，请用一句话回复用户：{user_msg}"
    ).content
    return {"messages": [AIMessage(reply, name="draft")]}


def compliance_check(state: State):
    # 内部第二步：合规审查，回复缺承诺时效则补标准话术
    if "工作日" not in state["messages"][-1].content:
        return {"messages": [AIMessage("退款将在 3 个工作日内原路退回。", name="compliance")]}
    return {}


refund_builder = StateGraph(State)
refund_builder.add_node("draft_reply", draft_reply)
refund_builder.add_node("compliance_check", compliance_check)
refund_builder.add_edge(START, "draft_reply")
refund_builder.add_edge("draft_reply", "compliance_check")
refund_builder.add_edge("compliance_check", END)
refund_agent = refund_builder.compile()   # 必须先编译


# ---- 父图：总控路由 ----
def route(state: State) -> Command[Literal["refund_agent", "general_reply"]]:
    text = state["messages"][-1].content
    return Command(goto="refund_agent" if "退款" in text else "general_reply")


def general_reply(state: State):
    return {"messages": [AIMessage("您好，我是通用客服，请问有什么可以帮您？")]}


builder = StateGraph(State)
builder.add_node("route", route)
builder.add_node("refund_agent", refund_agent)   # 编译后的子图直接作节点
builder.add_node("general_reply", general_reply)
builder.add_edge(START, "route")
builder.add_edge("general_reply", END)
graph = builder.compile()   # route 用 Command 跳转，无需显式出边

result = graph.invoke({"messages": [HumanMessage("耳机有质量问题，我要退款")]})
for msg in result["messages"][1:]:
    print(f"[{msg.name}] {msg.content}")

很抱歉给您带来不好的体验，耳机有质量问题可以退款，请您提供订单号和故障照片/视频，我马上为您处理。

退款将在 3 个工作日内原路退回。

## 子图可以有私有 key

客服场景延伸：退款专员子图还要登记**工单号**。工单号只属于专员内部流程（不该混进用户可见的
`messages`），于是放进子图私有 key `ticket_id`——父图 schema 里没有它，永远看不到。

In [5]:
class RefundState(TypedDict):
    messages: Annotated[list, add_messages]   # 与父图共享
    ticket_id: str                            # 子图私有：工单号


def open_ticket(state: RefundState):
    # 工单号只写私有 key，不进 messages（用户不该看到内部工单）
    return {"ticket_id": "T-2026-0920-001"}


refund_builder = StateGraph(RefundState)
refund_builder.add_node("open_ticket", open_ticket)
refund_builder.add_node("draft_reply", draft_reply)
refund_builder.add_node("compliance_check", compliance_check)
refund_builder.add_edge(START, "open_ticket")
refund_builder.add_edge("open_ticket", "draft_reply")
refund_builder.add_edge("draft_reply", "compliance_check")
refund_builder.add_edge("compliance_check", END)
refund_agent = refund_builder.compile()

builder = StateGraph(State)
builder.add_node("refund_agent", refund_agent)
builder.add_edge(START, "refund_agent")
graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage("耳机有质量问题，我要退款")]})
for msg in result["messages"][1:]:
    print(f"[{msg.name}] {msg.content}")

# ticket_id 只在子图内部流转：父图结果里没有这个 key
print("父图可见的 key:", list(result.keys()))

您好，非常抱歉耳机出现质量问题，请提供订单号及问题照片/视频，我会尽快为您核实并办理退款。

退款将在 3 个工作日内原路退回。

父图可见的 key:
['messages']

## 用 `subgraphs=True` 观察子图内部

真实需求：运营/前端要展示"专员正在做什么"（已建工单 → 已生成回复 → 已过合规），
或排查时定位卡在哪一步。默认 `stream` 只看到子图整体一个节点；加 `subgraphs=True`，
chunk 变成 `(namespace, chunk)`——namespace 定位到子图内部节点。

In [6]:
# 沿用上一格的 graph；ns=() 是父图层，ns=('refund_agent:<uuid>',) 是子图内部
for ns, chunk in graph.stream(
    {"messages": [HumanMessage("耳机有质量问题，我要退款")]},
    subgraphs=True,
):
    print(f"ns={ns} chunk={chunk}")

ns=('refund_agent:1239123e-4a46-65d7-0ee9-d188ba2da55c',) chunk={'open_ticket': {'ticket_id': 'T-2026-0920-001'}}

ns=('refund_agent:1239123e-4a46-65d7-0ee9-d188ba2da55c',) chunk={'draft_reply': {'messages': 
[AIMessage(content='很抱歉耳机出现质量问题，请您提供订单号，我将立即为您办理退款。', additional_kwargs={}, 
response_metadata={}, name='draft', id='0f275200-68fe-4227-99f7-c67ff2249010', tool_calls=[], 
invalid_tool_calls=[])]}}

ns=('refund_agent:1239123e-4a46-65d7-0ee9-d188ba2da55c',) chunk={'compliance_check': {'messages': 
[AIMessage(content='退款将在 3 个工作日内原路退回。', additional_kwargs={}, response_metadata={}, 
name='compliance', id='4c1a217f-117d-4d6b-a9fb-6e0b00b7cf2f', tool_calls=[], invalid_tool_calls=[])]}}

ns=() chunk={'refund_agent': {'messages': [HumanMessage(content='耳机有质量问题，我要退款', additional_kwargs={}, 
response_metadata={}, id='426b8cf9-0daf-4326-bcdd-bee2854a1684'), 
AIMessage(content='很抱歉耳机出现质量问题，请您提供订单号，我将立即为您办理退款。', additional_kwargs={}, 
response_metadata={}, name='draft', id='0f275200-68fe-4227-99f7-c67ff2249010', tool_calls=[], 
invalid_tool_calls=[]), AIMessage(content='退款将在 3 个工作日内原路退回。', additional_kwargs={}, 
response_metadata={}, name='compliance', id='4c1a217f-117d-4d6b-a9fb-6e0b00b7cf2f', tool_calls=[], 
invalid_tool_calls=[])]}}

## 实测结论

1. **同名 key 双向自动传递**：子图读写 `messages`，父图无感知拿到全部更新，全程零包装代码。
2. **子图私有 key 只在内部流转**（实测：`ticket_id` 从未出现在父图 state 中）。
3. **1.2.11 必须传编译后的图**：`add_node("x", 未编译builder)` 直接 `TypeError`。
4. 默认 `stream` 中子图节点以父图节点名平铺出现；`subgraphs=True` 才能看到子图内部节点与命名空间（uuid 每次运行会变）。